# SNF Approval Queue, Explainability, And Thresholds

**Executive takeaway:** The administrator-safe approval queue turns model and rule signals into a short weekly checklist: what to review, which source to check, and what action to take before payroll approval.

In [1]:
from common.plots import LetsPlot, aes, geom_bar, ggplot, labs, theme_minimal

from payroll_anomaly_ranking.columns import PayrollCol, ReviewCol
from payroll_anomaly_ranking.config import PayrollConfig
from payroll_anomaly_ranking.pipeline import run_pipeline
from payroll_anomaly_ranking.presentation import compact_case_cards

LetsPlot.setup_html()

results = run_pipeline(
    PayrollConfig(employee_count=160, pay_periods=12, review_budgets=(10, 25)),
)
queue = results.analyst_review_queue

## Administrator-Safe Approval Queue

The queue excludes synthetic evaluation labels and uses review-safe wording. It does not claim confirmed misconduct, fraud, or payroll error.

In [2]:
queue.select(
    [
        ReviewCol.RANK,
        PayrollCol.FACILITY_ID,
        PayrollCol.UNIT,
        PayrollCol.ROLE,
        PayrollCol.SHIFT_DATE,
        PayrollCol.SHIFT_TYPE,
        ReviewCol.APPROVAL_RISK_CATEGORY,
        ReviewCol.RECOMMENDED_ACTION,
        ReviewCol.SOURCE_TO_CHECK,
        ReviewCol.PRIMARY_REASON,
        ReviewCol.DOLLARS_AT_RISK,
    ],
).head(15)

rank,facility_id,unit,role,shift_date,shift_type,approval_risk_category,recommended_action,source_to_check,primary_reason,dollars_at_risk
u32,str,str,str,date,str,str,str,str,str,f64
1,"""SNF-F001""","""Long Term Care""","""Med Aide""",2024-06-20,"""Double""","""review before approval""","""Confirm schedule""","""Schedule""","""Paid hours materially exceed s…",372.07
2,"""SNF-F001""","""Long Term Care""","""Dietary""",2024-06-16,"""Double""","""review before approval""","""Verify timeclock edit""","""Timeclock""","""Overtime is unusually high for…",272.23
3,"""SNF-F001""","""Long Term Care""","""LPN""",2024-06-17,"""Double""","""review before approval""","""Verify timeclock edit""","""Timeclock""","""Overtime is unusually high for…",284.875
4,"""SNF-F001""","""Skilled Nursing""","""Dietary""",2024-06-14,"""Double""","""confirm if time permits""","""Approve known staffing excepti…","""Pay code""","""Gross pay materially differs f…",249.19
5,"""SNF-F001""","""Skilled Nursing""","""CNA""",2024-06-17,"""Double""","""confirm if time permits""","""Approve known staffing excepti…","""Pay code""","""Gross pay materially differs f…",204.39
…,…,…,…,…,…,…,…,…,…,…
11,"""SNF-F001""","""Skilled Nursing""","""RN""",2024-06-08,"""Night""","""monitor""","""Approve known staffing excepti…","""Pay code""","""Gross pay materially differs f…",163.96
12,"""SNF-F001""","""Skilled Nursing""","""Dietary""",2024-06-15,"""Day""","""monitor""","""Approve known staffing excepti…","""Pay code""","""High combined approval excepti…",32.99
13,"""SNF-F001""","""Long Term Care""","""LPN""",2024-06-11,"""Day""","""monitor""","""Approve known staffing excepti…","""Pay code""","""Gross pay materially differs f…",134.12


## Compact Case Cards

Case cards are designed for administrators, business office managers, DON/scheduling partners, or regional operators who need concise evidence before payroll approval.

In [3]:
compact_case_cards(queue, limit=5)

rank,employee_id,facility_id,unit,role,shift_date,shift_type,approval_risk_category,recommended_action,source_to_check,primary_reason,gross_pay,scheduled_hours,paid_hours,premium_pay,dollars_at_risk,explanation
u32,str,str,str,str,date,str,str,str,str,str,f64,f64,f64,f64,f64,str
1,"""SYN-SNF-E00039""","""SNF-F001""","""Long Term Care""","""Med Aide""",2024-06-20,"""Double""","""review before approval""","""Confirm schedule""","""Schedule""","""Paid hours materially exceed s…",582.07,16.0,17.76,0.0,372.07,"""Review before weekly SNF payro…"
2,"""SYN-SNF-E00047""","""SNF-F001""","""Long Term Care""","""Dietary""",2024-06-16,"""Double""","""review before approval""","""Verify timeclock edit""","""Timeclock""","""Overtime is unusually high for…",413.48,16.0,16.13,68.55,272.23,"""Review before weekly SNF payro…"
3,"""SYN-SNF-E00037""","""SNF-F001""","""Long Term Care""","""LPN""",2024-06-17,"""Double""","""review before approval""","""Verify timeclock edit""","""Timeclock""","""Overtime is unusually high for…",465.35,16.0,16.43,36.97,284.875,"""Review before weekly SNF payro…"
4,"""SYN-SNF-E00082""","""SNF-F001""","""Skilled Nursing""","""Dietary""",2024-06-14,"""Double""","""confirm if time permits""","""Approve known staffing excepti…","""Pay code""","""Gross pay materially differs f…",432.37,16.0,15.09,33.95,249.19,"""Review before weekly SNF payro…"
5,"""SYN-SNF-E00110""","""SNF-F001""","""Skilled Nursing""","""CNA""",2024-06-17,"""Double""","""confirm if time permits""","""Approve known staffing excepti…","""Pay code""","""Gross pay materially differs f…",387.73,16.0,15.5,34.88,204.39,"""Review before weekly SNF payro…"


## Facility Approval Summary

Facility summaries let leaders see where the queue is concentrated before drilling into individual shifts.

In [4]:
results.facility_approval_summary.sort("estimated_exposure", descending=True)

pay_period_index,facility_id,total_shifts,total_gross_pay,total_paid_hours,overtime_hours,premium_dollars,queue_count,high_priority_count,estimated_exposure,top_reason_categories,approval_readiness
i64,str,u32,f64,f64,f64,f64,i64,i64,f64,str,str
12,"""SNF-F004""",152,35037.66,1309.61,109.85,2727.17,25,5,6112.214148,"""High combined approval excepti…","""Likely supportable if source c…"
12,"""SNF-F001""",160,36224.85,1334.46,68.76,2660.83,25,3,6017.809479,"""High combined approval excepti…","""Likely supportable if source c…"
12,"""SNF-F005""",140,33757.66,1165.78,58.16,2081.52,25,2,4485.966126,"""High combined approval excepti…","""Likely supportable if source c…"
12,"""SNF-F003""",95,20739.46,784.43,32.79,1611.22,25,2,3713.390907,"""High combined approval excepti…","""Likely supportable if source c…"
12,"""SNF-F006""",110,29904.6,907.99,38.23,1732.99,25,2,3578.534086,"""High combined approval excepti…","""Likely supportable if source c…"
12,"""SNF-F002""",84,19830.84,693.53,30.86,1180.97,25,2,3144.086752,"""High combined approval excepti…","""Likely supportable if source c…"


In [5]:
summary = results.facility_approval_summary
(
    ggplot(summary, aes(PayrollCol.FACILITY_ID, "queue_count"))
    + geom_bar(stat="identity", fill="#c46f38")
    + labs(
        title="Latest-period approval queue count by facility",
        x="Facility",
        y="Queued records",
    )
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7fdefc75f790>)

## Weekly Operating Model

1. Review high-priority records before approval.
2. Check the named source: schedule, timeclock, pay code, pay policy, facility assignment, or employee lifecycle.
3. Approve known staffing exceptions when supported.
4. Escalate questionable payroll-code or lifecycle records.
5. Capture feedback for future calibration.

## What This Proves

The automated queue translates technical signals into administrator actions while preserving privacy and avoiding confirmed-error language.